# IS 477 Project Code

In [33]:
import pandas as pd
import numpy as np

In [34]:
dfCrashes = pd.read_csv("data/raw/crashes.csv")
dfPeople = pd.read_csv("data/raw/people.csv")

/tmp/ipykernel_2962/896320708.py:2: DtypeWarning: Columns (0: vehicle_id, 1: age, 2: bac_result, 3: cell_phone_use, 4: bac_result_value) have mixed types. Specify dtype option on import or set low_memory=False.
  dfPeople = pd.read_csv("data/raw/people.csv")


## Data Cleaning - Ganga

In [35]:
print("Our crashes dataset has", dfCrashes.shape[1], "columns and", dfCrashes.shape[0], "rows.")
print("Our people dataset has", dfPeople.shape[1], "columns and", dfPeople.shape[0], "rows.")

Our crashes dataset has 20 columns and 1041273 rows.
Our people dataset has 16 columns and 2285353 rows.


In [36]:
dfMain = dfCrashes.merge(dfPeople, left_on = "crash_record_id", right_on = "crash_record_id")

At this point, we considered working with a smaller sample of the data for quicker computing using `dfMainSample = dfMain.sample(frac = 0.1, random_state = 42)`, but since pandas can generally handle datasets with around 200,000 rows, we decided to continue with the full dataset.

In [44]:
dfMain.info()

<class 'pandas.DataFrame'>
RangeIndex: 2285353 entries, 0 to 2285352
Data columns (total 35 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   crash_record_id          str           
 1   crash_date               datetime64[us]
 2   posted_speed_limit       int64         
 3   traffic_control_device   str           
 4   weather_condition        str           
 5   lighting_condition       str           
 6   roadway_surface_cond     str           
 7   road_defect              str           
 8   crash_type               str           
 9   damage                   str           
 10  prim_contributory_cause  str           
 11  num_units                int64         
 12  injuries_total           float64       
 13  injuries_fatal           float64       
 14  injuries_incapacitating  float64       
 15  crash_hour               int64         
 16  crash_day_of_week        int64         
 17  crash_month              int64        

We notice

In [43]:
dfMain.head()

,crash_record_id,crash_date,posted_speed_limit,traffic_control_device,weather_condition,lighting_condition,roadway_surface_cond,road_defect,crash_type,damage,...,safety_equipment,airbag_deployed,ejection,injury_classification,driver_action,driver_vision,physical_condition,bac_result,cell_phone_use,bac_result_value
0,000013b0123279411e0ec856dae95ab9f0851764350b7f...,2020-11-16 13:50:00,35,NO CONTROLS,CLEAR,DAYLIGHT,DRY,UNKNOWN,NO INJURY / DRIVE AWAY,"$501 - $1,500",...,SAFETY BELT USED,DID NOT DEPLOY,NONE,NO INDICATION OF INJURY,IMPROPER PARKING,UNKNOWN,NORMAL,TEST NOT OFFERED,NaN,NaN
1,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,2016-04-16 05:49:00,30,TRAFFIC SIGNAL,CLEAR,DAWN,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,"OVER $1,500",...,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NONE,NO INDICATION OF INJURY,IMPROPER LANE CHANGE,UNKNOWN,UNKNOWN,TEST NOT OFFERED,NaN,NaN
2,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,2016-04-16 05:49:00,30,TRAFFIC SIGNAL,CLEAR,DAWN,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,"OVER $1,500",...,SAFETY BELT USED,DID NOT DEPLOY,NONE,NO INDICATION OF INJURY,NONE,NOT OBSCURED,NORMAL,TEST NOT OFFERED,NaN,NaN
3,000043c6564ec4d54bc4efd957d97ca97f38a965dd64b4...,2019-12-22 02:11:00,30,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,"OVER $1,500",...,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,UNKNOWN,NO INDICATION OF INJURY,UNKNOWN,UNKNOWN,UNKNOWN,TEST NOT OFFERED,NaN,NaN
4,00005696946846c8b8a1d378dba4e2a5ed84a9b2876fe0...,2024-02-02 09:48:00,30,NO CONTROLS,CLEAR,DAYLIGHT,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,"OVER $1,500",...,USAGE UNKNOWN,NOT APPLICABLE,NONE,NO INDICATION OF INJURY,EMERGENCY VEHICLE ON CALL,NOT OBSCURED,NORMAL,TEST NOT OFFERED,NaN,NaN


In [47]:
# calculate the percentage of missing values per column
missing = (dfMain.isna().sum() / len(dfMain) * 100).round(2)

# highlight columns with missing values, sorted from most to least missing
print(missing[missing > 0].sort_values(ascending = False))

bac_result_value         99.93
cell_phone_use           99.91
age                      41.41
driver_action            20.31
bac_result               20.21
driver_vision            19.54
physical_condition       19.46
injury_classification     3.96
sex                       2.88
airbag_deployed           2.69
vehicle_id                2.08
ejection                  1.98
latitude                  0.76
longitude                 0.76
safety_equipment          0.48
dtype: float64


In [40]:
# Convert crash_date to datetime
dfMain['crash_date'] = pd.to_datetime(dfMain['crash_date'])

In [41]:
# Convert age to numeric and cap at valid range
dfMain['age'] = pd.to_numeric(dfMain['age'], errors='coerce')
dfMain['age'] = dfMain['age'].where(dfMain['age'].between(0, 120))

In [42]:
print(dfMain['age'].describe())
print(dfMain[dfMain['age'] > 100]['age'].value_counts())

count    1.338914e+06
mean     3.798534e+01
std      1.708573e+01
min      0.000000e+00
25%      2.600000e+01
50%      3.500000e+01
75%      5.000000e+01
max      1.100000e+02
Name: age, dtype: float64
age
102.0    22
101.0    19
103.0    14
104.0     8
105.0     7
110.0     6
109.0     6
107.0     6
108.0     5
106.0     4
Name: count, dtype: int64


In [ ]:
print(dfMain['bac_result'].value_counts(dropna=False))

In [ ]:
print(dfMain['vehicle_id'].value_counts(dropna=False).head(10))

In [18]:
# drop useless columns
dfMainSample = dfMainSample.drop(columns=['cell_phone_use', 'bac_result_value'])

# convert age to numeric
dfMainSample['age'] = pd.to_numeric(dfMainSample['age'], errors='coerce')

# convert crash_date to datetime
dfMainSample['crash_date'] = pd.to_datetime(dfMainSample['crash_date'])

dfMainSample.info()

<class 'pandas.DataFrame'>
Index: 228535 entries, 164382 to 1935078
Data columns (total 33 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   crash_record_id          228535 non-null  str           
 1   crash_date               228535 non-null  datetime64[us]
 2   posted_speed_limit       228535 non-null  int64         
 3   traffic_control_device   228535 non-null  str           
 4   weather_condition        228535 non-null  str           
 5   lighting_condition       228535 non-null  str           
 6   roadway_surface_cond     228535 non-null  str           
 7   road_defect              228535 non-null  str           
 8   crash_type               228535 non-null  str           
 9   damage                   228535 non-null  str           
 10  prim_contributory_cause  228535 non-null  str           
 11  num_units                228535 non-null  int64         
 12  injuries_total           2

In [19]:
# see missing value percentages
missing = (dfMainSample.isna().sum() / len(dfMainSample) * 100).round(2)
print(missing[missing > 0].sort_values(ascending=False))

# see what non-numeric values were in age
dfMain_age_check = dfPeople['age'].value_counts(dropna=False).head(20)
print(dfMain_age_check)

age                      41.52
driver_action            20.33
bac_result               20.16
driver_vision            19.53
physical_condition       19.45
injury_classification     3.96
sex                       2.88
airbag_deployed           2.71
vehicle_id                2.08
ejection                  1.99
latitude                  0.78
longitude                 0.78
safety_equipment          0.47
dtype: float64
age
NaN                 548438
USAGE UNKNOWN       169937
SAFETY BELT USED    158286
27.0                 34345
25.0                 34294
26.0                 33806
28.0                 33784
24.0                 33106
29.0                 32995
30.0                 31512
23.0                 31475
31.0                 30631
32.0                 29738
33.0                 28629
22.0                 28528
34.0                 26976
35.0                 26431
21.0                 26054
36.0                 25728
37.0                 24783
Name: count, dtype: int64


In [20]:
# filter out clearly bad age values (keep only reasonable ages)
dfMainSample['age'] = dfMainSample['age'].where(dfMainSample['age'].between(0, 100))

# check how many rows have ALL of the ~20% missing columns null
drivers_only = dfMainSample[['driver_action', 'driver_vision', 'physical_condition']].isna().all(axis=1).sum()
print(f"Rows missing all driver columns: {drivers_only}")

print(dfMainSample['person_type'].value_counts())

Rows missing all driver columns: 44216
person_type
DRIVER                 177768
PASSENGER               46030
PEDESTRIAN               2754
BICYCLE                  1725
NON-MOTOR VEHICLE         217
NON-CONTACT VEHICLE        41
Name: count, dtype: int64


In [21]:
dfMainSample['bac_result'].value_counts(dropna=False).head(10)

bac_result
TEST NOT OFFERED                   155976
NaN                                 46079
NONE                                 9216
TEST REFUSED                         1568
UNKNOWN                               717
TEST PERFORMED, RESULTS UNKNOWN       356
25                                    287
30                                    286
28                                    280
TEST TAKEN                            278
Name: count, dtype: int64

In [22]:
# replace numeric bac values with TEST TAKEN
import re
def clean_bac(val):
    if pd.isna(val):
        return val
    if re.match(r'^\d+\.?\d*$', str(val)):
        return 'TEST TAKEN'
    return val

dfMainSample['bac_result'] = dfMainSample['bac_result'].apply(clean_bac)
dfMainSample['bac_result'].value_counts(dropna=False).head(10)

bac_result
TEST NOT OFFERED                   155976
NaN                                 46079
TEST TAKEN                          14588
NONE                                 9216
TEST REFUSED                         1568
UNKNOWN                               717
TEST PERFORMED, RESULTS UNKNOWN       356
TOTALLY EJECTED                        24
TRAPPED/EXTRICATED                      6
PARTIALLY EJECTED                       5
Name: count, dtype: int64

In [23]:
valid_bac = ['TEST NOT OFFERED', 'NONE', 'TEST REFUSED', 'UNKNOWN', 
             'TEST PERFORMED, RESULTS UNKNOWN', 'TEST TAKEN']

dfMainSample['bac_result'] = dfMainSample['bac_result'].where(
    dfMainSample['bac_result'].isin(valid_bac), other=None
)

dfMainSample['bac_result'].value_counts(dropna=False)

bac_result
TEST NOT OFFERED                   155976
NaN                                 46114
TEST TAKEN                          14588
NONE                                 9216
TEST REFUSED                         1568
UNKNOWN                               717
TEST PERFORMED, RESULTS UNKNOWN       356
Name: count, dtype: int64

In [24]:
cat_columns = ['traffic_control_device', 'weather_condition', 'lighting_condition',
               'roadway_surface_cond', 'road_defect', 'crash_type', 'damage',
               'prim_contributory_cause', 'person_type', 'sex', 'safety_equipment',
               'airbag_deployed', 'ejection', 'injury_classification', 'driver_action',
               'driver_vision', 'physical_condition', 'bac_result']

dfMainSample[cat_columns] = dfMainSample[cat_columns].astype('category')

dfMainSample.info()

<class 'pandas.DataFrame'>
Index: 228535 entries, 164382 to 1935078
Data columns (total 33 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   crash_record_id          228535 non-null  str           
 1   crash_date               228535 non-null  datetime64[us]
 2   posted_speed_limit       228535 non-null  int64         
 3   traffic_control_device   228535 non-null  category      
 4   weather_condition        228535 non-null  category      
 5   lighting_condition       228535 non-null  category      
 6   roadway_surface_cond     228535 non-null  category      
 7   road_defect              228535 non-null  category      
 8   crash_type               228535 non-null  category      
 9   damage                   228535 non-null  category      
 10  prim_contributory_cause  228535 non-null  category      
 11  num_units                228535 non-null  int64         
 12  injuries_total           2